# Lab 7 · Notebook 3: A grid, and a loose end

**Time:** about 40 minutes. **What you will do:** put several panels in one figure, write a plotting function you can reuse, and find out who the 18 skin lines that do not need SOX10 are.

As in notebook 2, the self-checks look at the figure you made. Keep the names they ask for: `fig` for the figure, and `axes` for a grid of panels.

In [ ]:
# Run this first. It loads the DepMap table and makes skin and rest.
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

FILE = "depmap.csv"
URL = "https://raw.githubusercontent.com/HocheggerLab/y3-bio-python/main/lab07/data/" + FILE

if not Path(FILE).exists():
    urllib.request.urlretrieve(URL, FILE)

df = pd.read_csv(FILE)
skin = df[df["oncotree_lineage"] == "Skin"]
rest = df[df["oncotree_lineage"] != "Skin"]
print("all:", len(df), "  skin:", len(skin), "  rest:", len(rest))

## 1. Two panels side by side

`plt.subplots(1, 2)` makes one row of two panels. You get back the figure and `axes`, which holds both panels. `axes[0]` is the left panel and `axes[1]` the right one, counting from 0 as with a list. Each of them is an `ax` like the ones in notebook 2, so everything you learned works on it.

In [ ]:
# DEMO - a histogram on the left, a box plot on the right
fig, axes = plt.subplots(1, 2)
fig.set_size_inches(10, 4)

axes[0].hist(df["SOX10"], bins=40)
axes[0].set_xlabel("SOX10 gene effect")

axes[1].boxplot([skin["SOX10"], rest["SOX10"]], tick_labels=["skin", "all others"])
axes[1].set_ylabel("SOX10 gene effect")
plt.show()

### Broken

This should draw a histogram in the left panel. Read the last line of the error: it says what kind of object `axes` is.

In [ ]:
# BROKEN - a histogram in the left panel
fig, axes = plt.subplots(1, 2)
axes[0].hist(df["BRAF"], bins=40)
plt.show()


# --- self-check (do not edit) ---
assert len(axes[0].patches) == 40, "Draw the histogram on axes[0]"
print("Fixed. axes is an array of panels, not one panel. Pick one with axes[0] before you draw on it.")

### Write it

**Your task:** make a figure with two panels side by side (`fig, axes`).

- **Left:** the scatter plot from notebook 2: BRAF against SOX10, the rest in grey, skin in colour on top.
- **Right:** a histogram of SOX10 in the skin lines, with 20 bins.

Give each panel a title.

In [ ]:
# TODO - scatter on the left, histogram on the right
fig, axes = plt.subplots(1, 2)
fig.set_size_inches(10, 4)

axes[0].scatter(rest["BRAF"], rest["SOX10"], s=6, alpha=0.3, color="grey", label="other lineages")
axes[0].scatter(skin["BRAF"], skin["SOX10"], s=12, color="tab:blue", label="skin")
axes[0].set_xlabel("BRAF gene effect")
axes[0].set_ylabel("SOX10 gene effect")
axes[0].set_title("Skin lines need both genes")
axes[0].legend()

axes[1].hist(skin["SOX10"], bins=20)
axes[1].set_xlabel("SOX10 gene effect")
axes[1].set_title("SOX10 in skin")
plt.show()

# --- self-check (do not edit) ---
assert axes.shape == (2,), f"axes has the shape {axes.shape}. Use plt.subplots(1, 2)"
sizes = [len(c.get_offsets()) for c in axes[0].collections]
assert sizes == [1092, 73], f"The left panel has dot layers of {sizes}. Draw rest first, then skin"
assert len(axes[1].patches) == 20, f"The right panel has {len(axes[1].patches)} bars. Use bins=20"
assert sum(b.get_height() for b in axes[1].patches) == 73, "The right panel should show the 73 skin lines"
assert axes[0].get_title() and axes[1].get_title(), "Give each panel a title with set_title"
print("Correct: two panels in one figure")

## 2. A grid of panels

`plt.subplots(2, 2)` makes two rows of two panels. Now `axes` is a grid, and you pick a panel with **two** numbers: the row, then the column, both counting from 0.

| | column 0 | column 1 |
|---|---|---|
| **row 0** | `axes[0][0]` | `axes[0][1]` |
| **row 1** | `axes[1][0]` | `axes[1][1]` |

In [ ]:
# DEMO - which panel is which?
fig, axes = plt.subplots(2, 2)
print("axes has the shape", axes.shape)
axes[0][0].set_title("axes[0][0]")
axes[0][1].set_title("axes[0][1]")
axes[1][0].set_title("axes[1][0]")
axes[1][1].set_title("axes[1][1]")
fig.tight_layout()
plt.show()

`fig.tight_layout()` moves the panels apart so that titles and labels do not overlap.

### Write it

**Your task:** make a 2 × 2 grid with a histogram of SOX10 for one lineage in each panel, with 20 bins, written out panel by panel (no loop yet):

| | column 0 | column 1 |
|---|---|---|
| **row 0** | Skin | Lung |
| **row 1** | Bowel | Breast |

Give each panel the lineage name as its title.

In [ ]:
# TODO - four histograms, one per panel
fig, axes = plt.subplots(2, 2)

axes[0][0].hist(df[df["oncotree_lineage"] == "Skin"]["SOX10"], bins=20)
axes[0][0].set_title("Skin")
axes[0][1].hist(df[df["oncotree_lineage"] == "Lung"]["SOX10"], bins=20)
axes[0][1].set_title("Lung")
axes[1][0].hist(df[df["oncotree_lineage"] == "Bowel"]["SOX10"], bins=20)
axes[1][0].set_title("Bowel")
axes[1][1].hist(df[df["oncotree_lineage"] == "Breast"]["SOX10"], bins=20)
axes[1][1].set_title("Breast")

fig.tight_layout()
plt.show()

# --- self-check (do not edit) ---
assert axes.shape == (2, 2), f"axes has the shape {axes.shape}. Use plt.subplots(2, 2)"
expected = {(0, 0): ("Skin", 73), (0, 1): ("Lung", 124), (1, 0): ("Bowel", 62), (1, 1): ("Breast", 53)}
for (r, c), (name, n) in expected.items():
    panel = axes[r][c]
    assert panel.get_title() == name, f"axes[{r}][{c}] has the title {panel.get_title()!r}, expected {name!r}"
    total = sum(b.get_height() for b in panel.patches)
    assert total == n, f"axes[{r}][{c}] shows {total:.0f} cell lines, but {name} has {n}"
print("Correct: four lineages, four panels")

## 3. Visit every panel with a loop

That was a lot of copying. `axes.flat` lets a `for` loop visit the panels one by one, row by row: `axes[0][0]`, `axes[0][1]`, `axes[1][0]`, `axes[1][1]`.

To walk through the panels and a list of names together, use `zip`. It pairs the first panel with the first name, the second with the second, and so on.

In [ ]:
# DEMO - zip pairs things up
for number, name in zip([1, 2, 3], ["Skin", "Lung", "Bowel"]):
    print(number, name)

In [ ]:
# DEMO - the same grid of titles, with a loop
names = ["Skin", "Lung", "Bowel", "Breast"]
fig, axes = plt.subplots(2, 2)
for ax, name in zip(axes.flat, names):
    ax.set_title(name)
fig.tight_layout()
plt.show()

### Write it

**Your task:** redraw your four histograms from section 2 with a loop over `zip(axes.flat, names)`. Inside the loop, filter `df` to the lineage, draw its SOX10 histogram with 20 bins, and set the title.

In [ ]:
# TODO - four histograms from a loop
names = ["Skin", "Lung", "Bowel", "Breast"]
fig, axes = plt.subplots(2, 2)
for ax, name in zip(axes.flat, names):
    lineage = df[df["oncotree_lineage"] == name]
    ax.hist(lineage["SOX10"], bins=20)
    ax.set_title(name)
fig.tight_layout()
plt.show()

# --- self-check (do not edit) ---
assert axes.shape == (2, 2), "Use plt.subplots(2, 2)"
titles = [ax.get_title() for ax in axes.flat]
assert titles == names, f"The titles are {titles}. Set each panel's title to its lineage"
totals = [sum(b.get_height() for b in ax.patches) for ax in axes.flat]
assert totals == [73, 124, 62, 53], f"The panels show {totals} cell lines. Filter df to each lineage inside the loop"
print("Correct: the same figure from four lines of loop")

## 4. Shared axes change what the reader sees

Look at your four histograms again. Each panel chose its **own** x range to fit its data. So the skin bars and the lung bars look about the same, even though the skin values run down to -3 and the lung values hardly go below -0.5. A reader comparing panels by eye would miss the most important difference.

`sharex=True` gives every panel the same x range. `sharey=True` does the same for the y axis.

### Write it

**Your task:** draw the same four histograms with the loop, but this time with `plt.subplots(2, 2, sharex=True)`. Compare the two figures.

In [ ]:
# TODO - the same grid with a shared x axis
fig, axes = plt.subplots(2, 2, sharex=True)
for ax, name in zip(axes.flat, names):
    lineage = df[df["oncotree_lineage"] == name]
    ax.hist(lineage["SOX10"], bins=20)
    ax.set_title(name)
fig.tight_layout()
plt.show()

# --- self-check (do not edit) ---
assert axes.shape == (2, 2), "Use plt.subplots(2, 2, sharex=True)"
assert axes[0][0].get_shared_x_axes().joined(axes[0][0], axes[1][1]), "The panels do not share their x axis. Add sharex=True"
assert [ax.get_title() for ax in axes.flat] == names, "Set each panel's title to its lineage"
print("Correct: now the skin panel stands out at once")

Whenever the panels of a figure show the same measurement, share that axis. Otherwise every panel gets its own scale and the comparison is lost.

## 5. A plotting function that takes an `ax`

In notebook 2 you wrote the "box with dots" plot several times. Put it in a function and you write it once.

Give the function the **panel to draw on** as an argument. Then the same function can fill a single figure, the left panel of a pair, or every panel of a grid:

```python
def plot_gene(ax, gene):
    ...draw on ax, never on plt...
```

As with `most_specific` in notebook 1, the function should use its arguments: `gene` for the column, never a fixed name like `"SOX10"`.

### Write it

**Your task:** complete `plot_gene(ax, gene)`. On `ax` it should draw:

1. a box plot of `gene` for `skin` and `rest`, named `skin` and `all others`, with `showfliers=False`
2. every cell line as a dot: skin around x = 1, the rest around x = 2
3. the title `gene`, and the y label `gene effect`

The check calls your function on a new figure with a gene it chooses.

In [ ]:
# TODO - complete the function
def plot_gene(ax, gene):
    """Box plot of one gene, skin against all others, with every cell line as a dot."""
    ax.boxplot([skin[gene], rest[gene]], tick_labels=["skin", "all others"], showfliers=False)
    ax.scatter(np.random.normal(1, 0.06, len(skin)), skin[gene], s=8, alpha=0.6)
    ax.scatter(np.random.normal(2, 0.06, len(rest)), rest[gene], s=4, alpha=0.3)
    ax.set_title(gene)
    ax.set_ylabel("gene effect")

# --- self-check (do not edit) ---
fig, ax = plt.subplots()
plot_gene(ax, "BRAF")
assert ax.get_title() == "BRAF", f"The title is {ax.get_title()!r}. Use ax.set_title(gene)"
assert [t.get_text() for t in ax.get_xticklabels()] == ["skin", "all others"], "Name the boxes skin and all others"
layers = {len(c.get_offsets()): c.get_offsets()[:, 1] for c in ax.collections}
assert sorted(layers) == [73, 1092], f"Your dot layers have {sorted(layers)} dots. Draw skin and rest"
assert np.allclose(sorted(layers[73]), sorted(skin["BRAF"])), "The skin dots are not BRAF values. Use skin[gene], not a fixed gene name"
assert ax.get_ylabel() == "gene effect", "Set the y label to 'gene effect'"
plt.close(fig)
print("Correct: plot_gene draws any gene on any panel")

### Write it

Back to the trap from notebook 1: RAN topped every list, but it is not specific to skin. Your function can show that.

**Your task:** make a 2 × 2 grid with `sharey=True`, and use a loop with `plot_gene` to draw one gene from `genes` in each panel. Add a title over the whole figure with `fig.suptitle`.

In [ ]:
# TODO - four genes, one function
genes = ["SOX10", "BRAF", "RAN", "KRAS"]
fig, axes = plt.subplots(2, 2, sharey=True)
fig.set_size_inches(9, 7)
for ax, gene in zip(axes.flat, genes):
    plot_gene(ax, gene)
fig.suptitle("Needed in skin, or needed everywhere?")
fig.tight_layout()
plt.show()

# --- self-check (do not edit) ---
assert axes.shape == (2, 2), "Use plt.subplots(2, 2, sharey=True)"
assert [ax.get_title() for ax in axes.flat] == genes, f"The panel titles are {[ax.get_title() for ax in axes.flat]}. Call plot_gene(ax, gene) in a loop over zip(axes.flat, genes)"
assert axes[0][0].get_shared_y_axes().joined(axes[0][0], axes[1][1]), "Add sharey=True so the four genes are on one scale"
assert fig.get_suptitle(), "Add a title over the figure with fig.suptitle(...)"
print("Correct: SOX10 and BRAF split skin from the rest. RAN is low on both sides.")

The figure says what notebook 1 said with numbers. SOX10 and BRAF: the skin box sits well below the rest. RAN: both boxes sit far down, because every cell needs it. KRAS: skin needs it **less** than the others do.

### Broken

A second function, to draw a histogram of any gene. Every panel shows the same histogram, and the titles claim four different genes. Find the line that ignores the argument.

In [ ]:
# BROKEN - four genes, but one histogram
def plot_hist(ax, gene):
    ax.hist(df[gene], bins=30)
    ax.set_title(gene)


fig, axes = plt.subplots(2, 2, sharex=True)
for ax, gene in zip(axes.flat, ["SOX10", "BRAF", "RAN", "KRAS"]):
    plot_hist(ax, gene)
fig.tight_layout()
plt.show()


# --- self-check (do not edit) ---
left_edges = [ax.patches[0].get_x() for ax in axes.flat]
assert len(set(left_edges)) == 4, "The four panels still show the same data"
print("Fixed. The function used the column 'SOX10' whatever gene it was given. Use the argument: df[gene].")

## 6. The loose end: who are the 18?

Notebook 1 found 18 skin lines with SOX10 above -0.5: they hardly need it. In the box plots they are the dots sitting level with the other cancers. Are they a mistake in the data, or are they telling you something?

Start with the column you have not used yet: `oncotree_primary_disease`, the type of cancer.

### Write it

**Your task:**

1. Make a table `exceptions` with the skin lines whose `SOX10` is above `-0.5`.
2. Store `exceptions["oncotree_primary_disease"].value_counts()` in `disease_counts`, and look at it.

In [ ]:
# TODO - what kind of cancer are the 18?
exceptions = skin[skin["SOX10"] > -0.5]
disease_counts = exceptions["oncotree_primary_disease"].value_counts()
disease_counts

# --- self-check (do not edit) ---
for name in ["exceptions", "disease_counts"]:
    assert name in globals(), f"Make a variable called {name}"
assert len(exceptions) == 18, f"exceptions should have 18 rows, got {len(exceptions)}. Filter skin with skin['SOX10'] > -0.5"
assert disease_counts.to_dict() == {"Melanoma": 10, "Cutaneous Squamous Cell Carcinoma": 5, "Merkel Cell Carcinoma": 3}, f"disease_counts is {disease_counts.to_dict()}"
print("Correct: 10 melanomas, 5 squamous cell carcinomas, 3 Merkel cell carcinomas")

So the 18 split into two groups, with two different explanations.

**Eight are not melanoma at all.** `oncotree_lineage` says where a cancer grew, not which cell it came from. Squamous cell carcinoma comes from keratinocytes, the cells that make up most of the skin's surface. Merkel cell carcinoma comes from Merkel cells, which sense touch. Neither starts from a pigment cell, so there is no reason for either to need SOX10. "Skin" was hiding three different diseases.

**Ten are melanomas that do not need SOX10.** This is real biology. Some melanoma cells switch off the programme that makes them pigment cells, SOX10 included, and grow without it. Researchers call this **phenotype switching**, and it is one way melanomas escape treatment.

### Write it

Now ask the question about melanoma, wherever it grew. `.str.contains("Melanoma")` is `True` for every row whose text contains the word, which catches skin melanoma, melanoma of the eye and the rest.

**Your task:**

1. Make a table `melanoma` of every cell line whose `oncotree_primary_disease` contains `Melanoma`, and store its length in `n_melanoma`.
2. Store the number of those lines with `SOX10` below `-0.5` in `n_melanoma_need`.

In [ ]:
# TODO - melanoma, wherever it grew
melanoma = df[df["oncotree_primary_disease"].str.contains("Melanoma")]
n_melanoma = len(melanoma)
n_melanoma_need = len(melanoma[melanoma["SOX10"] < -0.5])
print(melanoma["oncotree_lineage"].value_counts())

# --- self-check (do not edit) ---
for name in ["melanoma", "n_melanoma", "n_melanoma_need"]:
    assert name in globals(), f"Make a variable called {name}"
assert n_melanoma == 74, f"n_melanoma should be 74, got {n_melanoma}. Filter df (not skin) with .str.contains('Melanoma')"
assert n_melanoma_need == 62, f"n_melanoma_need should be 62, got {n_melanoma_need}. Filter melanoma with SOX10 < -0.5"
print("Correct: 62 of the 74 melanoma lines need SOX10, from the skin, the eye and elsewhere")

That is the answer from the end of the lecture: SOX10 is needed by **melanoma**, not by **skin**. The lineage column was a good start, and the disease column was the better question.

## 7. One figure for the whole result

**Your task:** make one figure with three panels in a row (`fig, axes = plt.subplots(1, 3)`) that a reader could follow without you there:

1. **Left:** a histogram of SOX10 in all cell lines. *Some lines need SOX10.*
2. **Middle:** the grey-and-colour scatter of BRAF against SOX10. *They are skin lines, and they need BRAF too.*
3. **Right:** a box plot of SOX10 in the skin lines, one box per disease in `diseases`, with every line as a dot. *Within skin, it is the melanomas.*

Give every panel a title and axis labels. Use `fig.set_size_inches(15, 4)` so the panels have room. The box names are long: `tick_labels=["Melanoma", "Squamous", "Merkel"]` keeps them short.

In [ ]:
# TODO - the result in three panels
diseases = ["Melanoma", "Cutaneous Squamous Cell Carcinoma", "Merkel Cell Carcinoma"]
groups = [skin[skin["oncotree_primary_disease"] == d]["SOX10"] for d in diseases]

fig, axes = plt.subplots(1, 3)
fig.set_size_inches(15, 4)

axes[0].hist(df["SOX10"], bins=40)
axes[0].set_xlabel("SOX10 gene effect")
axes[0].set_ylabel("Number of cell lines")
axes[0].set_title("A few lines need SOX10")

axes[1].scatter(rest["BRAF"], rest["SOX10"], s=6, alpha=0.3, color="grey", label="other lineages")
axes[1].scatter(skin["BRAF"], skin["SOX10"], s=12, color="tab:blue", label="skin")
axes[1].set_xlabel("BRAF gene effect")
axes[1].set_ylabel("SOX10 gene effect")
axes[1].set_title("They are skin lines, and need BRAF too")
axes[1].legend()

axes[2].boxplot(groups, tick_labels=["Melanoma", "Squamous", "Merkel"], showfliers=False)
for i, values in enumerate(groups, start=1):
    axes[2].scatter(np.random.normal(i, 0.06, len(values)), values, s=8, alpha=0.6)
axes[2].set_xlabel("Skin cancer type")
axes[2].set_ylabel("SOX10 gene effect")
axes[2].set_title("Within skin, it is the melanomas")

fig.tight_layout()
fig.savefig("sox10_story.png", dpi=300, bbox_inches="tight")
plt.show()

# --- self-check (do not edit) ---
assert axes.shape == (3,), f"axes has the shape {axes.shape}. Use plt.subplots(1, 3)"
assert len(axes[0].patches) > 0 and sum(b.get_height() for b in axes[0].patches) == 1165, "The left panel should be a histogram of SOX10 in all 1,165 lines"
assert [len(c.get_offsets()) for c in axes[1].collections] == [1092, 73], "The middle panel should draw rest first, then skin"
assert len(axes[2].get_xticklabels()) == 3, "The right panel should have three boxes, one per disease"
assert sorted(len(c.get_offsets()) for c in axes[2].collections) == [3, 5, 65], "Put every skin line on its box as a dot: 65, 5 and 3 lines"
for i, ax in enumerate(axes):
    assert ax.get_title(), f"axes[{i}] has no title"
    assert ax.get_xlabel() and ax.get_ylabel(), f"axes[{i}] needs an x label and a y label"
print("Correct: three panels a reader can follow. Save it with fig.savefig if you have not already.")

## If you finish early

1. Add a fourth panel: SOX10 for **all** melanoma lines, one box per `oncotree_lineage` (Skin, Eye, Vulva/Vagina). Do eye melanomas need SOX10 as well?
2. The 10 melanomas that do not need SOX10: do they still need BRAF? Compare the mean BRAF effect of `exceptions` that are melanoma with the other skin melanomas.
3. Change `plot_gene` so it takes the two groups as arguments too: `plot_gene(ax, gene, group_a, group_b, names)`. Use it to compare melanoma with every other cell line.

In [ ]:
# Your code here

## 🔭 Explore further

- Look up SOX10 and BRAF yourself in the DepMap portal: <https://depmap.org/portal/>
- Search for **melanoma phenotype switching** to read about the 10 melanomas that do not need SOX10.
- The matplotlib gallery, for when you want a plot you have not seen yet: <https://matplotlib.org/stable/gallery/index.html>

## Recap

You can now:

- make a row of panels with `plt.subplots(1, n)` and pick one with `axes[i]`
- make a grid with `plt.subplots(rows, cols)` and pick a panel with `axes[row][col]`
- visit every panel with `for ax, name in zip(axes.flat, names)`
- use `sharex=True` or `sharey=True` so that panels can be compared by eye
- write a plotting function that takes the panel as an argument and uses its arguments only
- check whether a group label (here, "Skin") is hiding more than one kind of thing

You started this lab with a table of 1,165 cell lines. You finish it knowing which gene melanoma needs, why some skin lines do not need it, and how to show both in one figure.